# HiRAG-Ontology — Knowledge Graph Explorer

This notebook provides interactive visualisation and analysis of the knowledge graph
constructed by the multi-agent pipeline from Russian oncological clinical guidelines.

**Requirements:** Run `py -m evaluation.run_eval` first to build the graph.

In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter, defaultdict

# Load knowledge graph
with open('results/knowledge_graph_final.json', encoding='utf-8') as f:
    kg_data = json.load(f)

print(f'Entities: {len(kg_data["entities"])}')
print(f'Relations: {len(kg_data["relations"])}')

In [ ]:
# Build NetworkX graph
G = nx.DiGraph()

entity_map = {e['id']: e for e in kg_data['entities']}

for e in kg_data['entities']:
    G.add_node(e['id'],
               label=e['label'],
               entity_type=e['entity_type'])

for r in kg_data['relations']:
    G.add_edge(r['subject_id'], r['object_id'],
               predicate=r['predicate'],
               weight=r['confidence'])

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Is DAG: {nx.is_directed_acyclic_graph(G)}')
print(f'Weakly connected components: {nx.number_weakly_connected_components(G)}')

In [ ]:
# === GRAPH STATISTICS ===
degrees = dict(G.degree())
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())

print('=== Graph Statistics ===')
print(f'Average degree:     {np.mean(list(degrees.values())):.2f}')
print(f'Max degree:         {max(degrees.values())} ({entity_map[max(degrees, key=degrees.get)]["label"]})')
print(f'Graph density:      {nx.density(G):.6f}')

# PageRank
pagerank = nx.pagerank(G, alpha=0.85, max_iter=200)
top_pr = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]
print('\n=== Top 10 by PageRank ===')
for eid, score in top_pr:
    label = entity_map[eid]['label']
    etype = entity_map[eid]['entity_type']
    print(f'  {score:.4f}  {label} ({etype})')

In [ ]:
# === FIG 1: ENTITY TYPE DISTRIBUTION ===
type_counts = Counter(e['entity_type'] for e in kg_data['entities'])

TYPE_COLORS = {
    'Condition':          '#534AB7',
    'LabTest':            '#185FA5',
    'Other':              '#888780',
    'Procedure':          '#1D9E75',
    'Drug':               '#D85A30',
    'AnatomicalStructure':'#378ADD',
    'DosageRegimen':      '#BA7517',
    'Organization':       '#D4537E',
    'Symptom':            '#639922',
}

fig, ax = plt.subplots(figsize=(10, 5))
labels = list(type_counts.keys())
values = list(type_counts.values())
colors = [TYPE_COLORS.get(l, '#888') for l in labels]

bars = ax.barh(labels, values, color=colors, height=0.6, edgecolor='none')
for bar, val in zip(bars, values):
    pct = val / sum(values) * 100
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({pct:.1f}%)', va='center', fontsize=9)

ax.set_xlabel('Number of entities')
ax.set_title('Entity Type Distribution in Knowledge Graph', fontweight='bold')
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('results/nb_entity_types.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === FIG 2: TOP ENTITIES BY DEGREE ===
top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:20]
top_labels = [entity_map[n]['label'] for n, _ in top_nodes]
top_degrees = [d for _, d in top_nodes]
top_colors = [TYPE_COLORS.get(entity_map[n]['entity_type'], '#888') for n, _ in top_nodes]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(top_labels))
bars = ax.bar(x, top_degrees, color=top_colors, edgecolor='none')
for bar, val in zip(bars, top_degrees):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(top_labels, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Degree (connections)')
ax.set_title('Top 20 Entities by Graph Degree', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)

legend = [mpatches.Patch(color=c, label=t)
          for t, c in TYPE_COLORS.items()
          if any(entity_map[n]['entity_type'] == t for n, _ in top_nodes)]
ax.legend(handles=legend, fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig('results/nb_top_entities.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === FIG 3: NETWORKX GRAPH VISUALISATION (top 30 nodes) ===
# Select top 30 nodes by degree
top30 = [n for n, _ in sorted(degrees.items(),
          key=lambda x: x[1], reverse=True)[:30]]
subG = G.subgraph(top30).copy()

pos = nx.spring_layout(subG, k=2.5, seed=42, iterations=100)

node_colors = [TYPE_COLORS.get(
    entity_map[n]['entity_type'], '#888') for n in subG.nodes()]
node_sizes = [degrees[n] * 20 + 200 for n in subG.nodes()]
node_labels = {n: entity_map[n]['label'] for n in subG.nodes()}

# Edge colors by predicate
PRED_COLORS = {
    'treats':        '#1D9E75',
    'diagnosed_by':  '#185FA5',
    'related_to':    '#AAAAAA',
    'part_of':       '#BA7517',
    'causes':        '#D85A30',
}
edge_colors = [PRED_COLORS.get(
    subG[u][v]['predicate'], '#ccc') for u, v in subG.edges()]

fig, ax = plt.subplots(figsize=(16, 11))
ax.set_facecolor('#FAFAFA')

nx.draw_networkx_edges(subG, pos, ax=ax,
    edge_color=edge_colors, alpha=0.5,
    arrows=True, arrowsize=12,
    connectionstyle='arc3,rad=0.1',
    node_size=node_sizes)

nx.draw_networkx_nodes(subG, pos, ax=ax,
    node_color=node_colors, node_size=node_sizes,
    alpha=0.9, linewidths=0.5, edgecolors='white')

nx.draw_networkx_labels(subG, pos, node_labels, ax=ax,
    font_size=7, font_color='white', font_weight='bold')

# Legends
node_legend = [mpatches.Patch(color=c, label=t)
               for t, c in TYPE_COLORS.items()
               if any(entity_map[n]['entity_type'] == t
                      for n in subG.nodes())]
from matplotlib.lines import Line2D
edge_legend = [Line2D([0],[0], color=c, lw=2, label=p)
               for p, c in PRED_COLORS.items()]

leg1 = ax.legend(handles=node_legend, title='Entity type',
                 loc='upper left', fontsize=8, framealpha=0.9)
ax.add_artist(leg1)
ax.legend(handles=edge_legend, title='Relation type',
          loc='lower left', fontsize=8, framealpha=0.9)

ax.set_title(
    'Knowledge Graph — Top 30 Entities by Degree\n'
    f'(node size ∝ degree · {subG.number_of_nodes()} nodes · '
    f'{subG.number_of_edges()} edges shown)',
    fontsize=13, fontweight='bold', pad=16)
ax.axis('off')
plt.tight_layout()
plt.savefig('results/nb_graph_viz.png', dpi=180, bbox_inches='tight')
plt.show()
print('Graph saved to results/nb_graph_viz.png')

In [ ]:
# === FIG 4: PREDICATE DISTRIBUTION ===
pred_counts = Counter(r['predicate'] for r in kg_data['relations'])

fig, ax = plt.subplots(figsize=(8, 4))
preds = list(pred_counts.keys())
counts = list(pred_counts.values())
colors = [PRED_COLORS.get(p, '#888') for p in preds]

bars = ax.barh(preds, counts, color=colors, edgecolor='none', height=0.6)
for bar, val in zip(bars, counts):
    pct = val / sum(counts) * 100
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            f'{val} ({pct:.1f}%)', va='center', fontsize=10)

ax.set_xlabel('Number of relations')
ax.set_title('Predicate Distribution in Knowledge Graph', fontweight='bold')
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('results/nb_predicates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === SUBGRAPH QUERY: find neighbourhood of a specific entity ===
QUERY_ENTITY = 'ОЛЛ'  # Change this to any entity label

# Find entity by label
target = None
for e in kg_data['entities']:
    if e['label'] == QUERY_ENTITY:
        target = e['id']
        break

if target:
    # Get 2-hop neighbourhood
    neighbors_1 = set(G.successors(target)) | set(G.predecessors(target))
    neighbors_2 = set()
    for n in neighbors_1:
        neighbors_2 |= set(G.successors(n)) | set(G.predecessors(n))
    
    subgraph_nodes = {target} | neighbors_1
    subG2 = G.subgraph(subgraph_nodes).copy()
    
    print(f"Entity: {QUERY_ENTITY}")
    print(f"1-hop neighbours: {len(neighbors_1)}")
    print(f"Subgraph: {subG2.number_of_nodes()} nodes, "
          f"{subG2.number_of_edges()} edges")
    
    # Print all direct relations
    print(f"\nDirect relations of '{QUERY_ENTITY}':")
    for u, v, data in G.edges(target, data=True):
        obj = entity_map[v]['label']
        print(f"  → [{data['predicate']}] → {obj}")
    for u, v, data in G.in_edges(target, data=True):
        subj = entity_map[u]['label']
        print(f"  ← [{data['predicate']}] ← {subj}")
else:
    print(f"Entity '{QUERY_ENTITY}' not found in graph")